# Démo 7 : pipeline CNN avec transforms et DataLoader

Objectif : construire le pipeline complet d'un CNN : transforms (normalisation et augmentation), Dataset, DataLoader, un petit CNN dont on suit les formes couche par couche, puis quelques époques sur GPU. Pour vos propres images rangées en dossiers, `ImageFolder` joue le rôle du Dataset (TP3).

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- préparer les images avec des transforms, dont l'augmentation de données ;
- construire un petit CNN, écrit comme une classe, et suivre la forme des données couche par couche ;
- l'entraîner sur GPU en suivant loss et accuracy sur l'entraînement et la validation, puis le mesurer une seule fois sur le test.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [1]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge le jeu d'entraînement, puis celui de
  test, s'ils ne sont pas déjà là ;
- `%%capture` : masque la barre de progression du téléchargement.

In [2]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Les transforms

**Les transforms préparent chaque image au moment où on la lit.**

Un réseau attend des tenseurs de nombres à virgule, centrés autour de 0. Et pour l'entraînement, on
ajoute de l'**augmentation** : de petites modifications aléatoires (rotation, miroir) qui donnent au modèle des
variantes des mêmes images, et l'aident à généraliser.

Le code :
- `v2.ToImage()`, `v2.ToDtype(torch.float32, scale=True)` : conversion en tenseur de valeurs entre 0 et 1 ;
- `v2.Normalize(mean=[0.5], std=[0.5])` : ramène les valeurs entre -1 et 1 ;
- `v2.RandomRotation(degrees=10)`, `v2.RandomHorizontalFlip(p=0.5)` : rotation de plus ou moins 10 degrés, et
  miroir une fois sur deux ;
- `v2.Compose([...])` : enchaîne les étapes ; `eval_tf` n'a pas d'augmentation.

In [3]:
import torch
from torchvision.transforms import v2

conversion = [v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(mean=[0.5], std=[0.5])]
train_tf = v2.Compose([v2.RandomRotation(degrees=10), v2.RandomHorizontalFlip(p=0.5)] + conversion)
eval_tf = v2.Compose(conversion)
print(train_tf)

`Compose` affiche les étapes dans l'ordre d'application : rotation de plus ou moins 10 degrés, flip horizontal, conversion en tenseur, passage en flottants entre 0 et 1, puis normalisation autour de 0.

**Piège fréquent** : appliquer l'augmentation au jeu de test ; on mesure alors des images qui n'existent pas.

## 3. Entraînement, validation, test

**Trois jeux, comme dans la démo 5 : entraînement, validation, test.**

Les Datasets appliquent la transform à chaque image lue. On garde 10 000 images d'entraînement pour la
validation, mesurée à chaque époque ; le test reste de côté jusqu'à la fin. Dans la démo 5, `random_split`
suffisait ; ici, la validation ne doit pas recevoir l'augmentation : on crée deux Datasets sur les mêmes
fichiers, l'un avec `train_tf`, l'autre avec `eval_tf`.

Le code :
- `torch.randperm(60000)` : les numéros des 60 000 images, dans un ordre aléatoire ;
- `Subset(..., indices[:50000])` : les 50 000 premiers numéros, avec `train_tf`, pour entraîner ;
- `Subset(..., indices[50000:])` : les 10 000 autres, avec `eval_tf`, pour valider ;
- `transform=eval_tf` pour la validation et le test : on n'augmente jamais les images qu'on mesure.

In [4]:
from torch.utils.data import Subset
from torchvision import datasets

torch.manual_seed(0)
indices = torch.randperm(60000)
train_ds = Subset(datasets.FashionMNIST("data", train=True, transform=train_tf), indices[:50000])
val_ds = Subset(datasets.FashionMNIST("data", train=True, transform=eval_tf), indices[50000:])
test_ds = datasets.FashionMNIST("data", train=False, transform=eval_tf)
print(len(train_ds), len(val_ds), len(test_ds))

50000 images pour entraîner, 10000 pour valider, et les 10000 de test mises de côté. Les indices mélangés sont coupés en deux : aucune image ne sert à la fois à l'entraînement et à la validation.

**Piège fréquent** : couper le jeu d'entraînement après lui avoir donné `train_tf` ; la validation hérite alors de l'augmentation.

## 4. Les DataLoader

**Des minibatchs de 64 images, mélangées pour l'entraînement.**

Le code :
- `DataLoader(train_ds, batch_size=64, shuffle=True)` : comme dans la démo 5 ;
- `val_loader`, `test_loader` : des lots de 256, sans mélange, car on ne fait que mesurer ;
- `next(iter(train_loader))` : un premier lot, pour voir sa forme et l'afficher.

In [5]:
from torch.utils.data import DataLoader

torch.manual_seed(0)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

Un minibatch : 64 images d'un canal de 28 x 28 pixels, et leurs 64 labels.

## 5. Un batch à l'écran

**Huit images du lot, avec leur classe.**

On vérifie que l'augmentation reste réaliste ; une image trop déformée n'appartiendrait plus à sa
classe.

Le code (à survoler) : une rangée de huit cases, chacune avec une image et le nom de sa classe.

In [6]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for k, ax in enumerate(axes):
    ax.imshow(images[k, 0], cmap="gray")
    ax.set_title(test_ds.classes[labels[k]], fontsize=8)
    ax.axis("off")
plt.show()

Huit images du batch, avec leur classe. L'augmentation se voit : vêtements légèrement tournés, certains retournés en miroir. Pour des vêtements, ces variations restent réalistes.

**Essayez** : relancez la cellule du DataLoader et celle-ci ; les mêmes vêtements n'ont plus exactement la même orientation.

## 6. Le modèle CNN

**Un réseau convolutif : deux blocs convolution, ReLU, pooling, puis une partie linéaire.**

Les blocs de convolution extraient des motifs, de plus en plus larges à chaque bloc ; la partie
linéaire combine ces motifs pour décider de la classe. On l'écrit comme une classe `nn.Module`, comme
`PetitReseau` dans la démo 4 : `__init__` déclare les blocs, `forward` décrit le calcul.

Le code :
- `self.bloc1` : `nn.Conv2d(1, 16, kernel_size=3, padding=1)`, 16 noyaux sur l'image d'un canal, puis
  `nn.ReLU()` et `nn.MaxPool2d(2)`, qui divise la taille par deux ;
- `self.bloc2` : le même schéma ; `nn.Conv2d(16, 8, ...)` reçoit les 16 cartes du premier bloc et en produit 8 ;
- `self.classifieur` : `nn.Flatten()` aplatit les cartes en un vecteur, puis `nn.Linear(8 * 7 * 7, 32)` et
  `nn.Linear(32, 10)` donnent les 10 scores ;
- `forward` : l'image traverse `bloc1`, puis `bloc2`, puis `classifieur`.

In [ ]:
import torch.nn as nn


class PetitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.bloc1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.bloc2 = nn.ModuleList([
            nn.Conv2d(16, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ])
        self.classifieur = nn.ModuleDict({
            "flatten": nn.Flatten(),
            "linear1": nn.Linear(8 * 7 * 7, 32),
            "relu": nn.ReLU(),
            "linear2": nn.Linear(32, 10)
        })

    def forward(self, x):
        x = self.bloc1(x)
        for layer in self.bloc2:
            x = layer(x)
        x = self.classifieur["flatten"](x)
        x = self.classifieur["linear1"](x)
        x = self.classifieur["relu"](x)
        x = self.classifieur["linear2"](x)
        return x

Rien à afficher : la classe est définie, comme `PetitReseau` dans la démo 4. Elle ne contient encore aucun poids ; c'est l'appel `PetitCNN()` qui les crée.

## 7. Créer le modèle

La graine fixe les poids de départ : chaque exécution part du même modèle.

In [ ]:
torch.manual_seed(0)
model = PetitCNN()

Rien à afficher : le modèle est créé, avec des poids de départ aléatoires, mais les mêmes à chaque exécution grâce à la graine.

## 8. Suivre les formes couche par couche

**On fait passer une image dans chaque couche et on affiche la forme obtenue.**

L'erreur la plus fréquente en construisant un CNN est une taille qui ne correspond pas à l'entrée de
la couche linéaire. Suivre les formes montre d'où vient le `8 * 7 * 7`.

Le code :
- `x = images[:1]` : une seule image, en gardant la dimension du lot ;
- `for bloc in [model.bloc1, model.bloc2, model.classifieur]:` : les trois blocs, dans l'ordre de `forward` ;
- `for couche in bloc:` : chaque bloc est un `nn.Sequential`, qui se parcourt couche par couche ;
- `x = couche(x)` : on applique la couche, puis on affiche son nom et la forme de sa sortie.

Prédisez la taille après chaque pooling.

In [9]:
x = images[:1]
for bloc in [model.bloc1, model.bloc2, model.classifieur]:
    for couche in bloc.children():
        x = couche(x)
        print(f"{couche.__class__.__name__:10s} {tuple(x.shape)}")


Chaque convolution garde la taille, grâce au padding, et change le nombre de canaux (16, puis 8) ; chaque pooling divise la taille par deux : 28, 14, puis 7. D'où les 392 valeurs après `Flatten` : 8 canaux de 7 x 7. La dernière couche donne les 10 scores.

**Piège fréquent** : se tromper dans la taille d'entrée de la première couche linéaire ; le message d'erreur parle alors de formes incompatibles (`mat1 and mat2 shapes cannot be multiplied`).

## 9. Le modèle sur le GPU

**On envoie le modèle sur le GPU et on compte ses paramètres.**

Le code :
- `model.to(device)` : les poids du modèle vont sur le GPU ;
- `sum(p.numel() for p in model.parameters())` : le nombre total de paramètres.

Prédisez : plus ou moins de paramètres que le MLP de la démo 5 ?

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Le modèle a", sum(p.numel() for p in model.parameters()), "paramètres")

14226 paramètres seulement, contre plus de cent mille pour le MLP de la démo 5 : une convolution partage les mêmes poids sur toute l'image.

**À retenir** : les convolutions réutilisent les mêmes poids partout sur l'image ; c'est ce qui rend un CNN compact.

## 10. Mesurer la loss et l'accuracy (fourni)

**La fonction `evaluate` de la démo 5.**

Avec les deux mesures, on compare l'entraînement et la validation sur la loss comme sur l'accuracy.

Le code : mode évaluation, pas de gradient ; pour chaque lot, la loss multipliée par le nombre d'images du
lot (pour une moyenne sur tout le jeu) et le nombre de bonnes réponses ; la fonction renvoie la loss moyenne
et la part d'images bien classées.

In [11]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            scores = model(images)
            total_loss += loss_fn(scores, labels).item() * len(labels)
            correct += (scores.argmax(dim=1) == labels).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Rien à afficher : la fonction est définie. Elle renvoie deux nombres : la loss moyenne, puis l'accuracy.

## 11. Quelques époques

**La boucle de la démo 5, avec quatre mesures après chaque époque.**

On suit la loss et l'accuracy sur l'entraînement et sur la validation. La validation sert à suivre
l'apprentissage et à décider (nombre d'époques, augmentation) ; le test n'intervient pas.

Le code :
- `nn.CrossEntropyLoss()` et `Adam(..., lr=1e-3)` : la loss et l'optimiseur de la démo 5 ;
- pour chaque lot : prédiction, loss, `zero_grad()`, `backward()`, `step()` ;
- `evaluate(model, train_loader, loss_fn)` : la mesure sur l'entraînement, faite sur des images augmentées
  puisqu'elles passent par `train_tf` ;
- `evaluate(model, val_loader, loss_fn)` : la mesure sur la validation, sur des images non modifiées.

In [12]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1, 10):
    model.train()
    for images, labels in train_loader:
        loss = loss_fn(model(images.to(device)), labels.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_loader, loss_fn)
    val_loss, val_acc = evaluate(model, val_loader, loss_fn)
    print(
        f"époque {epoch}  loss {train_loss:.3f}  acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}"
    )

Une ligne par époque : `loss` et `acc` sur l'entraînement, `val_loss` et `val_acc` sur la validation. On lit les deux paires ensemble : tant que la validation progresse avec l'entraînement, le modèle apprend des motifs qui valent aussi pour des images qu'il n'a jamais vues.

**Essayez** : retirez l'augmentation de `train_tf` et comparez l'écart entre `acc` et `val_acc`.

## 12. Une seule mesure sur le jeu de test

Le modèle est fixé : on le mesure une seule fois sur des images qui n'ont servi ni à apprendre, ni à décider.

In [13]:
test_loss, test_acc = evaluate(model, test_loader, loss_fn)
print(f"test  loss {test_loss:.3f}  acc {test_acc:.3f}")

C'est le score à annoncer pour ce modèle : le jeu de test n'a servi ni à apprendre les poids, ni à choisir le nombre d'époques ou l'augmentation.

**À retenir** : la validation sert à suivre et à décider pendant l'entraînement ; le test se mesure une seule fois, à la fin.

## À retenir de cette démo

- Les transforms convertissent et normalisent les images ; l'augmentation les modifie au hasard, pour l'entraînement seulement.
- Un CNN enchaîne des blocs convolution, ReLU, pooling, puis `Flatten` et des couches linéaires ; suivre les formes couche par couche évite les erreurs de taille.
- La boucle est celle de la démo 5 ; on suit l'entraînement et la validation à chaque époque, et on mesure le test une seule fois, à la fin.